In [ ]:
# Import all libraries.
import os
import json
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from scipy.stats import pearsonr

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

In [ ]:
# Initialize Pandas settings.
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
pd.set_option("display.expand_frame_repr", False)

In [ ]:
# Enforce reproducablity.
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
# Using Google Drive:
from google.colab import drive
drive.mount('/content/drive/')
DATA_DIR = "/content/drive/MyDrive/CS 489/"
FILE = os.path.join(DATA_DIR, "CS 489 - Self Playing (New).csv")

os.makedirs(DATA_DIR + "models", exist_ok = True)
os.makedirs(DATA_DIR + "history/wally", exist_ok = True)

# # Using local directory:
# DATA_DIR = "/Users/xmastersteel/wally/dl/data"
# FILE = os.path.join(DATA_DIR, "CS 489 - Self Playing (New).csv")

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [ ]:

# Separate dataset into 75/10/15 split.
TRAIN_SPLIT = 0.75
VAL_SPLIT   = 0.10
TEST_SPLIT  = 0.15
df          = pd.read_csv(FILE)
train_parts, val_parts, test_parts = [], [], []

# Scan through all unique word pairs (30).
for pair_id, group in df.groupby("anchor_left (0)"):

    # Extract 15% of total dataset into test set.
    train_and_val, test = train_test_split(
        group, test_size = TEST_SPLIT, random_state = RANDOM_SEED)

    # Extract 10% of remaining 85% for validation split.
    train, val = train_test_split(
        train_and_val, test_size = VAL_SPLIT / (1 - TEST_SPLIT), random_state = RANDOM_SEED)

    train_parts.append(train)
    val_parts.append(val)
    test_parts.append(test)

# Turn train/val/test sets back into singular dfs.
train_df = pd.concat(train_parts).reset_index(drop = True)
val_df   = pd.concat(val_parts).reset_index(drop = True)
test_df  = pd.concat(test_parts).reset_index(drop = True)

# print(train_df)
# print(val_df)
# print(test_df)


In [ ]:

# Wavelength Dataset:
class WavelengthDataset(Dataset):
    def __init__(self, df, tokenizer, max_length = 64, augment = False):
        self.df = df.reset_index(drop = True)
        self.df["response_mean"] = pd.to_numeric(self.df["response_mean"], errors = "coerce")
        self.df = self.df.dropna(subset = ["response_mean"])
        self.tokenizer = tokenizer
        self.max_len   = max_length
        self.augment   = augment

    def __len__(self):
        return len(self.df) * (2 if self.augment else 1)

    def __getitem__(self, idx):
        flip   = self.augment and idx >= len(self.df)
        row    = self.df.iloc[idx % len(self.df)]
        left   = row["anchor_left (0)"]
        right  = row["anchor_right (100)"]
        label  = row["response_mean"] / 100.0

        # Doubles dataset, flipping anchor pairs.
        if flip:
            left, right = right, left
            label = 1.0 - label

        text = f"clue: {row['clue']} | left: {left} | right: {right}"
        enc  = self.tokenizer(
            text,
            max_length     = self.max_len,
            padding        = "max_length",
            truncation     = True,
            return_tensors = "pt"
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label":          torch.tensor(label, dtype = torch.float32)
        }

# Wally Model:
class Wally(nn.Module):
    def __init__(self, model_name, dropout = 0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)

        for param in self.encoder.parameters():
            param.requires_grad = False

        for layer in self.encoder.encoder.layer[-4:]:
            for param in layer.parameters():
                param.requires_grad = True

        hidden = self.encoder.config.hidden_size
        self.head = nn.Sequential(
            nn.Linear(hidden, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def mean_pool(self, token_embeddings, attention_mask):
        mask_expanded = attention_mask.unsqueeze(-1).float()
        return (token_embeddings * mask_expanded).sum(1) / mask_expanded.sum(1).clamp(min = 1e-9)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask = attention_mask)
        pooled = self.mean_pool(out.last_hidden_state, attention_mask)
        return self.head(pooled).squeeze(-1)

In [ ]:

EPOCH_EXPERIMENTS = [50, 100, 200]
BATCH_SIZE_EXPERIMENTS = [4, 16]
LR_EXPERIMENTS = [1e-5, 2e-5]

# Run through the 8 different experiments:
for epoch_exp in EPOCH_EXPERIMENTS:
  for bs_exp in BATCH_SIZE_EXPERIMENTS:
    for lr_exp in LR_EXPERIMENTS:
      print(f"Experiment - Epochs: {epoch_exp}, Batch Size: {bs_exp}, Learning Rate: {lr_exp}")

      # Hyperparamters Setup:
      MODEL_NAME    = "sentence-transformers/all-mpnet-base-v2"
      EPOCHS        = epoch_exp
      BATCH_SIZE    = bs_exp
      LR            = lr_exp
      SAVE_LOCATION = f"wally_{EPOCHS}epochs_{BATCH_SIZE}bs_{LR}lr"
      DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
      print(f"Training on: {DEVICE}")

      tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

      train_ds = WavelengthDataset(train_df, tokenizer, augment = True)
      val_ds   = WavelengthDataset(val_df,   tokenizer, augment = False)
      test_ds  = WavelengthDataset(test_df,  tokenizer, augment = False)

      train_loader = DataLoader(train_ds, batch_size = BATCH_SIZE, shuffle = True)
      val_loader   = DataLoader(val_ds,   batch_size = BATCH_SIZE)
      test_loader  = DataLoader(test_ds,  batch_size = BATCH_SIZE)

      wally     = Wally(MODEL_NAME).to(DEVICE)
      criterion = nn.L1Loss() # MAE Loss

      optimizer = AdamW(
          filter(lambda p: p.requires_grad, wally.parameters()),
          lr = LR, weight_decay = 1e-2
      )

      scheduler = get_linear_schedule_with_warmup(
          optimizer,
          num_warmup_steps   = len(train_loader) * 2,
          num_training_steps = len(train_loader) * EPOCHS
      )

      # Prepare directories for each experiment.
      os.makedirs("models/", exist_ok = True)

      # Prepare JSON output.
      history = {
          "info": {
              "model_name": MODEL_NAME.split("/")[-1],
              "epochs":     EPOCHS,
              "batch_size": BATCH_SIZE,
              "lr":         LR,
              "device":     str(DEVICE),
              "seed":       RANDOM_SEED
          }
      }

      # Train & Validation Evaluation
      best_val_mae    = float("inf")
      best_epoch_stats = {}

      # Ranges I extracted from physical Wavelength game.
      BLUE_RANGE_MAE   = 3.1
      ORANGE_RANGE_MAE = 9.4
      YELLOW_RANGE_MAE = 15.5

      epoch_history = []
      for epoch in range(EPOCHS):

          # Run Wally on training set.
          wally.train()
          train_loss = 0
          for batch in train_loader:
              ids    = batch["input_ids"].to(DEVICE)
              mask   = batch["attention_mask"].to(DEVICE)
              labels = batch["label"].to(DEVICE)

              optimizer.zero_grad()
              preds = wally(ids, mask)
              loss  = criterion(preds, labels)
              loss.backward()
              nn.utils.clip_grad_norm_(wally.parameters(), 1.0)
              optimizer.step()
              scheduler.step()
              train_loss += loss.item()


          # Run Wally on validation set.
          wally.eval()
          val_loss   = 0
          val_preds  = []
          val_labels = []
          with torch.no_grad():
              for batch in val_loader:
                  ids    = batch["input_ids"].to(DEVICE)
                  mask   = batch["attention_mask"].to(DEVICE)
                  labels = batch["label"].to(DEVICE)

                  preds     = wally(ids, mask)
                  val_loss += criterion(preds, labels).item()

                  preds_np  = preds.cpu().numpy()  * 100
                  labels_np = labels.cpu().numpy() * 100
                  val_preds.extend(preds_np)
                  val_labels.extend(labels_np)


          # Calculate the metrics for validation.
          val_preds_arr  = np.array(val_preds)
          val_labels_arr = np.array(val_labels)
          errors         = np.abs(val_preds_arr - val_labels_arr)

          val_mae  = np.mean(errors)
          val_r, _ = pearsonr(val_preds_arr, val_labels_arr)

          within_blue_acc   = np.mean(errors <= BLUE_RANGE_MAE)
          within_orange_acc = np.mean(errors <= ORANGE_RANGE_MAE)
          within_yellow_acc = np.mean(errors <= YELLOW_RANGE_MAE)

          epoch_history.append({
            "train_loss": float(train_loss / len(train_loader)),
            "val_loss":   float(val_loss   / len(val_loader)),
            "val_mae":    float(val_mae),
            "val_r":      float(val_r),
          })


          # Only save the best epoch's stats.
          if val_mae < best_val_mae:
              best_val_mae = val_mae
              torch.save(wally.state_dict(), f"{DATA_DIR}models/{SAVE_LOCATION}.pth")
              print(f" - New best model saved!")

              print(
                  f"Epoch {epoch + 1} | "
                  f"Train Loss: {train_loss/len(train_loader):.4f} | "
                  f"Val Loss: {val_loss/len(val_loader):.4f} | "
                  f"Val MAE: {val_mae:.2f} | Pearson r: {val_r:.3f} | "
                  f"Blue: {within_blue_acc * 100:.2f}% | Orange: {within_orange_acc * 100:.2f}% | Yellow: {within_yellow_acc * 100:.2f}%"
              )

              best_epoch_stats = {
                  "best_epoch": epoch + 1,
                  "train_loss": float(train_loss / len(train_loader)),
                  "val_loss":   float(val_loss   / len(val_loader)),
                  "val_mae":    float(val_mae),
                  "val_r":      float(val_r),
                  "within_range": {
                      "blue":   float(within_blue_acc),
                      "orange": float(within_orange_acc),
                      "yellow": float(within_yellow_acc),
                  },
                  "val_preds":  val_preds_arr.tolist(),
                  "val_labels": val_labels_arr.tolist(),
                  "val_errors": errors.tolist(),
              }

      # Update tentative JSON.
      history.update( {
          "best_val_mae": float(best_val_mae),
          "wavelength_ranges": {
              "blue_range_mae":   BLUE_RANGE_MAE,
              "orange_range_mae": ORANGE_RANGE_MAE,
              "yellow_range_mae": YELLOW_RANGE_MAE,
          },
          "best_epoch": best_epoch_stats,
          "epoch_history": epoch_history
      })

      # Test Evaluation
      wally.load_state_dict(torch.load(f"{DATA_DIR}models/{SAVE_LOCATION}.pth", map_location = DEVICE))
      wally.eval()

      test_preds, test_labels = [], []
      with torch.no_grad():
          for batch in test_loader:
              ids    = batch["input_ids"].to(DEVICE)
              mask   = batch["attention_mask"].to(DEVICE)
              preds  = wally(ids, mask).cpu().numpy() * 100
              labels = batch["label"].numpy()         * 100
              test_preds.extend(preds)
              test_labels.extend(labels)

      # Calculate metrics based on test data.
      test_preds_arr  = np.array(test_preds)
      test_labels_arr = np.array(test_labels)
      test_errors     = np.abs(test_preds_arr - test_labels_arr)
      test_mae        = np.mean(test_errors)
      test_r, _       = pearsonr(test_preds_arr, test_labels_arr)

      test_within_blue_acc   = np.mean(test_errors <= BLUE_RANGE_MAE)   * 100
      test_within_orange_acc = np.mean(test_errors <= ORANGE_RANGE_MAE) * 100
      test_within_yellow_acc = np.mean(test_errors <= YELLOW_RANGE_MAE) * 100

      print(f"\n---------- Final Test Results ----------")
      print(f"Test MAE:              {test_mae:.2f}")
      print(f"Pearson r:             {test_r:.3f}")
      print(f"Blue:   {test_within_blue_acc:.1f}% | Orange: {test_within_orange_acc:.1f}% | Yellow: {test_within_yellow_acc:.1f}%")

      # Create and add all information to JSON.
      history.update( {
          "test_mae": float(test_mae),
          "test_r":   float(test_r),
          "within_range": {
              "blue":   float(test_within_blue_acc),
              "orange": float(test_within_orange_acc),
              "yellow": float(test_within_yellow_acc),
          },
          "test_predictions": test_preds_arr.tolist(),
          "test_labels":      test_labels_arr.tolist(),
          "test_errors":      test_errors.tolist(),
      })

      # Create dir for experiment history, then add JSON.
      os.makedirs("history/wally", exist_ok = True)
      with open(f"{DATA_DIR}history/wally/{SAVE_LOCATION}.json", "w") as f:
          json.dump(history, f, indent = 4)

      print(f"Saved to history/wally/{SAVE_LOCATION}.json")



Experiment - Epochs: 50, Batch Size: 4, Learning Rate: 1e-05
Training on: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 - New best model saved!
Epoch 1 | Train Loss: 0.3162 | Val Loss: 0.3147 | Val MAE: 31.40 | Pearson r: 0.016 | Blue: 3.33% | Orange: 7.33% | Yellow: 10.67%
 - New best model saved!
Epoch 14 | Train Loss: 0.3156 | Val Loss: 0.3116 | Val MAE: 31.08 | Pearson r: 0.108 | Blue: 0.67% | Orange: 6.00% | Yellow: 16.00%
 - New best model saved!
Epoch 15 | Train Loss: 0.3059 | Val Loss: 0.2975 | Val MAE: 29.66 | Pearson r: 0.142 | Blue: 5.33% | Orange: 15.33% | Yellow: 24.00%
 - New best model saved!
Epoch 16 | Train Loss: 0.2892 | Val Loss: 0.2855 | Val MAE: 28.44 | Pearson r: 0.221 | Blue: 6.00% | Orange: 20.67% | Yellow: 30.00%
 - New best model saved!
Epoch 17 | Train Loss: 0.2768 | Val Loss: 0.2693 | Val MAE: 26.80 | Pearson r: 0.298 | Blue: 6.00% | Orange: 22.00% | Yellow: 37.33%
 - New best model saved!
Epoch 18 | Train Loss: 0.2601 | Val Loss: 0.2525 | Val MAE: 25.09 | Pearson r: 0.397 | Blue: 6.67% | Orange: 26.00% | Yellow: 44.67%
 - New best model saved!
Epoch 19 | Train Loss: 0.2446 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 - New best model saved!
Epoch 1 | Train Loss: 0.3162 | Val Loss: 0.3194 | Val MAE: 31.67 | Pearson r: -0.083 | Blue: 3.33% | Orange: 8.00% | Yellow: 12.00%
 - New best model saved!
Epoch 2 | Train Loss: 0.3164 | Val Loss: 0.3191 | Val MAE: 31.62 | Pearson r: -0.093 | Blue: 2.67% | Orange: 8.00% | Yellow: 12.67%
 - New best model saved!
Epoch 3 | Train Loss: 0.3165 | Val Loss: 0.3191 | Val MAE: 31.59 | Pearson r: -0.145 | Blue: 2.67% | Orange: 8.00% | Yellow: 12.00%
 - New best model saved!
Epoch 4 | Train Loss: 0.3164 | Val Loss: 0.3184 | Val MAE: 31.50 | Pearson r: -0.116 | Blue: 3.33% | Orange: 8.00% | Yellow: 11.33%
 - New best model saved!
Epoch 11 | Train Loss: 0.3161 | Val Loss: 0.3182 | Val MAE: 31.50 | Pearson r: 0.034 | Blue: 2.67% | Orange: 8.00% | Yellow: 11.33%
 - New best model saved!
Epoch 12 | Train Loss: 0.3160 | Val Loss: 0.3180 | Val MAE: 31.47 | Pearson r: 0.066 | Blue: 2.67% | Orange: 8.00% | Yellow: 10.67%
 - New best model saved!
Epoch 16 | Train Loss: 0.3159 | V

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 - New best model saved!
Epoch 1 | Train Loss: 0.3164 | Val Loss: 0.3171 | Val MAE: 31.33 | Pearson r: 0.026 | Blue: 3.33% | Orange: 7.33% | Yellow: 12.67%
 - New best model saved!
Epoch 11 | Train Loss: 0.3154 | Val Loss: 0.3147 | Val MAE: 31.03 | Pearson r: 0.192 | Blue: 2.67% | Orange: 6.67% | Yellow: 12.67%
 - New best model saved!
Epoch 12 | Train Loss: 0.3085 | Val Loss: 0.3104 | Val MAE: 30.64 | Pearson r: 0.203 | Blue: 2.67% | Orange: 7.33% | Yellow: 12.67%
 - New best model saved!
Epoch 13 | Train Loss: 0.2993 | Val Loss: 0.3016 | Val MAE: 29.12 | Pearson r: 0.205 | Blue: 2.67% | Orange: 12.67% | Yellow: 24.67%
 - New best model saved!
Epoch 14 | Train Loss: 0.2940 | Val Loss: 0.2991 | Val MAE: 28.73 | Pearson r: 0.221 | Blue: 6.67% | Orange: 15.33% | Yellow: 28.00%
 - New best model saved!
Epoch 16 | Train Loss: 0.2575 | Val Loss: 0.2619 | Val MAE: 25.05 | Pearson r: 0.404 | Blue: 5.33% | Orange: 26.00% | Yellow: 40.00%
 - New best model saved!
Epoch 17 | Train Loss: 0.2175 |